In [0]:
%pip install langchain langchain-openai databricks-langchain langchain-community langchain-databricks langgraph langchain_chroma -U ddgs youtube_search wikipedia
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_databricks import DatabricksEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [0]:

#data preprocessing
document=TextLoader("/Volumes/dev_agents/naval/raw/product-data (2) (1).txt").load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
chunks=text_splitter.split_documents(documents=document)

#embeddings 
embeddings=DatabricksEmbeddings(endpoint="databricks-gte-large-en")

# vectore store 
vectore_store=Chroma.from_documents(chunks,embeddings)

#serach or retriever
retriever=vectore_store.as_retriever()

#llm as brain
llm = ChatDatabricks(model="databricks-gpt-oss-20b")

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""You are an assistant for answering questions.
Use the provided context to respond. If the answer isn't clear, acknowledge that you don't know. Limit your response to three concise sentences.
 
Context: {context}
Question: {input}
"""
)

qa_chain=create_stuff_documents_chain(llm,prompt_template)
rag_chain=create_retrieval_chain(retriever,qa_chain)


print("Chat with your own Data")

question=input("what is your query?")

if question:
    response=rag_chain.invoke({"input":question})
    print(response['answer'])


In [0]:
from databricks_langchain import ChatDatabricks
from langchain_community.tools import YouTubeSearchTool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import Tool, initialize_agent
from langchain.prompts import PromptTemplate


In [0]:
search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_community.tools import YouTubeSearchTool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import Tool, initialize_agent
from langchain.prompts import PromptTemplate

import mlflow

# Enable automatic tracing for LangChain — all subsequent LangChain calls
# will be traced and logged to the active MLflow experiment automatically.
mlflow.langchain.autolog()


llm = ChatDatabricks(model="databricks-meta-llama-3-1-405b-instruct")

#define_tools


youtube_tool=Tool(
    name="youtube",
    func=YouTubeSearchTool(),
    description="youtube search"
)

search_tool=Tool(
    name="duck duck go",
    func=DuckDuckGoSearchRun(),
    description="duck duck go search"
)

tools=[youtube_tool,search_tool]

agent=initialize_agent(tools=tools,
                       llm=llm,
                       agent="zero-shot-react-description",
                       verbose=True
                       )


print("Hello, I am a chatbot. Ask me anything about a movie")
type=input("Enter the type of movie")

try:
    movie_type=f"Suggest a {type} movie."
    if movie_type:
        response=agent.run({"input":movie_type})
        print(response)

except Exception as e:
    print("An Error occurred",e)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_community.tools import YouTubeSearchTool, DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

import mlflow

mlflow.langchain.autolog()

# ── 1. LLM with streaming ───────────────────────────────────────────────────
llm = ChatDatabricks(
    model="databricks-claude-opus-4-5",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
    temperature=0.7,
)

# ── 2. Tools ────────────────────────────────────────────────────────────────

youtube_tool = Tool(
    name="YouTube Search",
    func=YouTubeSearchTool().run,
    description=(
        "Search YouTube for movie trailers, reviews, or clips. "
        "Use this when the user wants to watch something related to a movie."
    ),
)

web_search_tool = Tool(
    name="Web Search",
    func=DuckDuckGoSearchRun().run,
    description=(
        "Search the web for current movie information, box office results, "
        "release dates, or anything not covered by other tools."
    ),
)

wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=2))
wiki_tool = Tool(
    name="Wikipedia",
    func=wikipedia_tool.run,
    description=(
        "Look up detailed background on movies, directors, actors, genres, "
        "or film history. Great for in-depth factual information."
    ),
)

def imdb_search(query: str) -> str:
    return DuckDuckGoSearchRun().run(f"site:imdb.com {query}")

imdb_tool = Tool(
    name="IMDB Search",
    func=imdb_search,
    description=(
        "Search IMDB for ratings, cast, plot summaries, and reviews. "
        "Use this when the user asks about movie ratings or cast details."
    ),
)

tools = [youtube_tool, web_search_tool, wiki_tool, imdb_tool]

# ── 3. Personality ───────────────────────────────────────────────────────────
personality_prefix = """You are CineBot 🎬 — an enthusiastic, knowledgeable movie expert and critic.

Your personality:
- Passionate and opinionated about cinema, but always friendly
- Give concise, specific recommendations with reasons (not just titles)
- Mention IMDb ratings, directors, or standout performances when relevant
- If the user seems unsure, ask a clarifying question (mood, decade, language)
- Always offer to find a trailer on YouTube after a recommendation
- Use light movie references and wit, but never be condescending

You have access to YouTube, Wikipedia, IMDB, and Web Search — use them to give
accurate, up-to-date answers rather than relying solely on memory.

You have access to the following tools:"""

# ── 4. Agent ────────────────────────────────────────────────────────────────
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    agent_kwargs={"prefix": personality_prefix},
    verbose=True,
    handle_parsing_errors=True,
)

# ── 5. Chat loop ─────────────────────────────────────────────────────────────
print("\n🎬 CineBot is ready! Ask me anything about movies. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "bye"):
        print("CineBot: Lights out! See you at the next screening 🎥")
        break
    try:
        response = agent.run(user_input)
        print(f"\nCineBot: {response}\n")
    except Exception as e:
        print(f"CineBot: Hmm, something went wrong — {e}\n")